# sbdd-robust × Pocket2Mol — `real47` on Google Colab

**Runtime limits:** Full **47 × 5** Pocket2Mol jobs often take **many hours to days**. **Colab Free** typically disconnects in **~12 h**. Use **chunk indices** below, **Colab Pro+**, or a **cloud VM** you control for reliability.

**Faster?** Colab can give a **better GPU than a laptop** and avoids Windows PyG pain — but **not** if the session dies mid-run. Chunk + copy `data/results` to Drive after each chunk.

**Checkpoint file:** put `pretrained_Pocket2Mol.pt` on Drive (`MyDrive/pocket2mol_ckpt/`) or set `gdown` id in **Cell 2**.

In [ ]:
# @title Cell 0 — GPU + Python
import sys

try:
    import torch
    print("torch", torch.__version__)
    print("CUDA:", torch.cuda.is_available(), torch.version.cuda if torch.cuda.is_available() else "")
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("Install torch in next cell")
print("Python:", sys.version.split()[0])

In [ ]:
# @title Cell 1 — Install sbdd-robust + Pocket2Mol deps (Linux)
import os
import subprocess
import sys


def sh(cmd: str) -> None:
    print("$", cmd)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(cmd)


os.chdir("/content")

sh(f"{sys.executable} -m pip install -q -U pip")
try:
    import torch
except ImportError:
    sh(f"{sys.executable} -m pip install -q torch")
    import torch

tv = torch.__version__.split("+")[0]
cu = "cpu"
if torch.cuda.is_available():
    cv = torch.version.cuda.replace(".", "")
    cu = f"cu{cv}"
idx = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"
print("PyG wheel index:", idx)
sh(f"{sys.executable} -m pip install -q torch-scatter torch-sparse torch-cluster -f {idx}")
sh(f"{sys.executable} -m pip install -q torch-geometric==2.3.1")

sh(f"{sys.executable} -m pip install -q biopython pyyaml omegaconf pandas matplotlib seaborn tqdm easydict")
sh(f"{sys.executable} -m pip install -q rdkit lmdb")
sh(f"{sys.executable} -m pip install -q gdown")

print("Done base installs. Run next cell to clone sbdd-robust and pip install -e.")


In [ ]:
# @title Cell 2 — Clone repos + Pocket2Mol checkpoint
import os
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote


def git_clone(repo_url: str, dest: str) -> None:
    if os.path.isdir(dest):
        print("exists:", dest)
        return
    r = subprocess.run(["git", "clone", repo_url, dest], cwd="/content", capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-4000:])


os.chdir("/content")

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as _e:
    print("Drive mount:", _e)

# Pocket2Mol fork (sample_for_pdb.py)
P2M = Path("/content/Pocket2Mol")
if not P2M.is_dir():
    git_clone("https://github.com/pengxingang/Pocket2Mol.git", str(P2M))

# sbdd-robust (private: set GITHUB_TOKEN secret or env SBDD_ROBUST_REPO)
SBDD_ROOT = Path(os.environ.get("SBDD_ROBUST_ROOT", "/content/sbdd-robust")).resolve()
repo = os.environ.get("SBDD_ROBUST_REPO")
if not repo:
    try:
        from google.colab import userdata as ud

        tok = None
        for name in ("GITHUB_TOKEN", "GITHUB_1", "GH_TOKEN"):
            try:
                v = ud.get(name)
                if v and str(v).strip():
                    tok = str(v).strip()
                    print("git auth via secret", name)
                    break
            except Exception:
                continue
        if tok:
            repo = f"https://oauth2:{quote(tok, safe='')}@github.com/Marooncoloredchair/sbdd-robust.git"
    except Exception:
        pass
if not repo:
    repo = "https://github.com/Marooncoloredchair/sbdd-robust.git"

if (SBDD_ROOT / "pyproject.toml").is_file():
    print("Using sbdd-robust at", SBDD_ROOT)
elif not SBDD_ROOT.exists():
    git_clone(repo, str(SBDD_ROOT))
else:
    raise FileNotFoundError(f"{SBDD_ROOT} missing pyproject.toml — upload full repo")

os.environ["SBDD_ROBUST_ROOT"] = str(SBDD_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SBDD_ROOT)], check=True)

# Checkpoint: prefer Drive copy; else gdown file id
ckdir = P2M / "ckpt"
ckdir.mkdir(parents=True, exist_ok=True)
ckpt = ckdir / "pretrained_Pocket2Mol.pt"
DRIVE_CKPT = Path("/content/drive/MyDrive/pocket2mol_ckpt/pretrained_Pocket2Mol.pt")
GFILE_ID = os.environ.get("P2M_CKPT_GDRIVE_ID", "")  # optional: single-file id

if ckpt.is_file() and ckpt.stat().st_size > 1_000_000:
    print("Checkpoint OK:", ckpt)
elif DRIVE_CKPT.is_file():
    shutil.copy2(DRIVE_CKPT, ckpt)
    print("Copied from Drive:", ckpt)
elif GFILE_ID:
    import gdown

    gdown.download(id=GFILE_ID, output=str(ckpt), quiet=False)
else:
    raise FileNotFoundError(
        "Place pretrained_Pocket2Mol.pt in Drive at MyDrive/pocket2mol_ckpt/ "
        "or set env P2M_CKPT_GDRIVE_ID for gdown."
    )

print("ckpt:", ckpt.stat().st_size)

# PyTorch 2.6+ defaults torch.load(weights_only=True); Pocket2Mol checkpoints need full pickle.
_p2m_sf = P2M / "sample_for_pdb.py"
if _p2m_sf.is_file():
    _txt = _p2m_sf.read_text(encoding="utf-8")
    _needle = "ckpt = torch.load(config.model.checkpoint, map_location=args.device)"
    if _needle in _txt and "weights_only=False" not in _txt:
        _txt = _txt.replace(
            _needle,
            "try:\n        ckpt = torch.load(config.model.checkpoint, map_location=args.device, weights_only=False)\n    except TypeError:\n        ckpt = torch.load(config.model.checkpoint, map_location=args.device)",
        )
        _p2m_sf.write_text(_txt, encoding="utf-8")
        print("Patched Pocket2Mol sample_for_pdb.py for PyTorch 2.6+")


In [ ]:
# @title Cell 3 — Working directories
from pathlib import Path

PROJECT = Path("/content/sbdd_pocket2mol_work")
(PROJECT / "data/raw/real50").mkdir(parents=True, exist_ok=True)
(PROJECT / "data/results").mkdir(parents=True, exist_ok=True)
(PROJECT / "data/generations").mkdir(parents=True, exist_ok=True)
(PROJECT / "configs").mkdir(parents=True, exist_ok=True)
DRIVE_OUT = Path("/content/drive/MyDrive/sbdd-robust-results")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("PROJECT", PROJECT)
print("DRIVE_OUT", DRIVE_OUT)

In [ ]:
# @title Cell 4 — Download real47 PDBs + build Colab YAML (chunkable)
import os
import shutil
import sys
import urllib.error
import urllib.request
from pathlib import Path

import yaml

SBDD_ROOT = Path(os.environ.get("SBDD_ROBUST_ROOT", "/content/sbdd-robust")).resolve()
os.environ["SBDD_ROBUST_ROOT"] = str(SBDD_ROOT)
PROJECT = Path("/content/sbdd_pocket2mol_work")
RAW = PROJECT / "data/raw/real50"
RAW.mkdir(parents=True, exist_ok=True)

# If `pocket2mol_real47.yaml` is not in your GitHub clone, Colab will fetch/use fallbacks below.
REAL47_YAML_URL = ""  # @param {"type":"string"}
# Paste a raw link, e.g. https://raw.githubusercontent.com/<user>/sbdd-robust/main/configs/pocket2mol_real47.yaml
DRIVE_YAML = Path("/content/drive/MyDrive/sbdd-robust-data/configs/pocket2mol_real47.yaml")


def _load_yaml_text() -> str:
    candidates = [
        SBDD_ROOT / "configs" / "pocket2mol_real47.yaml",
        DRIVE_YAML,
    ]
    for p in candidates:
        if p.is_file():
            print("Using YAML:", p)
            return p.read_text(encoding="utf-8")
    urls = []
    if (REAL47_YAML_URL or "").strip():
        urls.append(REAL47_YAML_URL.strip())
    # Optional: same repo as Cell 2 default (edit branch if needed)
    urls.append(
        "https://raw.githubusercontent.com/Marooncoloredchair/sbdd-robust/main/configs/pocket2mol_real47.yaml"
    )
    for u in urls:
        try:
            print("Trying URL:", u)
            with urllib.request.urlopen(u, timeout=60) as r:
                return r.read().decode("utf-8")
        except urllib.error.HTTPError as e:
            print("  HTTP", e.code)
        except Exception as e:
            print("  error:", e)
    try:
        from google.colab import userdata as _ud

        _tok = None
        for _n in ("GITHUB_TOKEN", "GITHUB_1", "GH_TOKEN"):
            try:
                _v = _ud.get(_n)
                if _v and str(_v).strip():
                    _tok = str(_v).strip()
                    print("Trying GitHub Contents API with", _n)
                    break
            except Exception:
                continue
        if _tok:
            import base64
            import json

            _api = "https://api.github.com/repos/Marooncoloredchair/sbdd-robust/contents/configs/pocket2mol_real47.yaml?ref=main"
            _req = urllib.request.Request(
                _api,
                headers={
                    "Accept": "application/vnd.github+json",
                    "Authorization": f"Bearer {_tok}",
                    "X-GitHub-Api-Version": "2022-11-28",
                },
            )
            with urllib.request.urlopen(_req, timeout=60) as r:
                _pay = json.loads(r.read().decode("utf-8"))
            return base64.b64decode(_pay["content"]).decode("utf-8")
    except Exception as _api_e:
        print("GitHub API fallback:", _api_e)
    raise FileNotFoundError(
        "Cannot find pocket2mol_real47.yaml. Fix one of:\n"
        "  • Colab secret GITHUB_TOKEN (private repo), OR\n"
        "  • Upload to Drive: MyDrive/sbdd-robust-data/configs/pocket2mol_real47.yaml OR\n"
        "  • Set REAL47_YAML_URL (param above)."
    )


# --- Chunks (optional) ---
# CHUNKS_TOTAL=1, CHUNK_INDEX=0 ⇒ one session runs all pockets in the YAML.
# CHUNKS_TOTAL=3, CHUNK_INDEX=0|1|2 ⇒ split into thirds across Colab sessions (each gets its own run_id).
CHUNK_INDEX = 0  # @param {type:"integer"}
CHUNKS_TOTAL = 1  # @param {type:"integer"}

cfg = yaml.safe_load(_load_yaml_text())
pockets = list(cfg["pockets"])
n = len(pockets)
if CHUNKS_TOTAL < 1:
    raise ValueError("CHUNKS_TOTAL >= 1")
base = (n * CHUNK_INDEX) // CHUNKS_TOTAL
end = (n * (CHUNK_INDEX + 1)) // CHUNKS_TOTAL
cfg["pockets"] = pockets[base:end]
cfg["run_id"] = f"pocket2mol_real47_c{CHUNK_INDEX}of{CHUNKS_TOTAL}"

# Optional: reuse PDBs already on PC → copy from Drive folder matching **/real50/*.pdb
DRIVE_REAL50 = Path("/content/drive/MyDrive/sbdd-robust-data/raw/real50")
if DRIVE_REAL50.is_dir():
    for p in DRIVE_REAL50.glob("*.pdb"):
        shutil.copy2(p, RAW / p.name.lower())
    print("Copied PDBs from Drive:", DRIVE_REAL50)


def fetch_pdb(stem: str) -> None:
    dest = RAW / f"{stem.lower()}.pdb"
    if dest.is_file() and dest.stat().st_size > 1000:
        return
    url = f"https://files.rcsb.org/download/{stem.lower()}.pdb"
    urllib.request.urlretrieve(url, dest)


for row in cfg["pockets"]:
    stem = Path(row["pdb"]).stem.lower()
    fetch_pdb(stem)

cfg["project_root"] = "."
cfg["model"]["repo_root"] = "/content/Pocket2Mol"
cfg["model"]["checkpoint"] = "/content/Pocket2Mol/ckpt/pretrained_Pocket2Mol.pt"
cfg["model"]["python_exe"] = sys.executable
cfg["model"]["script_path"] = str(SBDD_ROOT / "scripts" / "pocket2mol_sample_drug_bridge.py")

out_y = PROJECT / "configs" / "pocket2mol_colab.yaml"
out_y.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Wrote", out_y)
print("Chunk pockets", base, "(incl)", end, "(excl)", "count", len(cfg["pockets"]), "run_id", cfg["run_id"])

In [ ]:
# @title Cell 5 — Run benchmark (long)
import os
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/content/sbdd_pocket2mol_work")
os.chdir(PROJECT)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Pocket2Mol imports
os.environ["PYTHONPATH"] = "/content/Pocket2Mol" + os.pathsep + os.environ.get("PYTHONPATH", "")

cmd = [sys.executable, "-u", "-m", "sbdd_robust", "run", "--config", "configs/pocket2mol_colab.yaml"]
print("Running:", " ".join(cmd))
p = subprocess.run(cmd, cwd=str(PROJECT))
if p.returncode != 0:
    raise RuntimeError(
        f"sbdd_robust exited {p.returncode}; scroll up. "
        "Unknown model.type pocket2mol → git pull and pip install -e sbdd-robust."
    )
print("Finished.")

In [ ]:
# @title Re-run benchmark — hardcoded command
# One-line shell equivalent (from /content):
#   cd /content/sbdd_pocket2mol_work && export PYTHONPATH=/content/Pocket2Mol:${PYTHONPATH:-} && python3 -u -m sbdd_robust run --config configs/pocket2mol_colab.yaml

import os
import subprocess
import sys
from pathlib import Path

SBDD_ROOT = Path(os.environ.get("SBDD_ROBUST_ROOT", "/content/sbdd-robust")).resolve()
if (SBDD_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(SBDD_ROOT), "pull", "--ff-only"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SBDD_ROOT)], check=True)

PROJECT = Path("/content/sbdd_pocket2mol_work")
os.chdir(PROJECT)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTHONPATH"] = "/content/Pocket2Mol" + os.pathsep + os.environ.get("PYTHONPATH", "")

cmd = [sys.executable, "-u", "-m", "sbdd_robust", "run", "--config", "configs/pocket2mol_colab.yaml"]
print("Running:", " ".join(cmd))
p = subprocess.run(cmd, cwd=str(PROJECT))
if p.returncode != 0:
    raise RuntimeError(f"sbdd_robust exited {p.returncode}; scroll up.")
print("Re-run finished OK.")


In [ ]:
# @title Cell 6 — Copy results to Drive + print brittleness_rate
import json
import shutil
from pathlib import Path

PROJECT = Path("/content/sbdd_pocket2mol_work")
DRIVE_OUT = Path("/content/drive/MyDrive/sbdd-robust-results")
res = PROJECT / "data/results"

for p in sorted(res.glob("*")):
    if p.is_file():
        dest = DRIVE_OUT / p.name
        shutil.copy2(p, dest)
        print("copied", p.name)
fig = res / "figures"
if fig.is_dir():
    for p in fig.glob("*.png"):
        d = DRIVE_OUT / "figures" / p.name
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, d)

meta = sorted(res.glob("run_meta__*.json"), key=lambda x: x.stat().st_mtime, reverse=True)
if meta:
    m = json.loads(meta[0].read_text())
    print("Latest meta:", meta[0].name)
    print("brittleness_rate:", m.get("brittleness_rate"))
else:
    print("No run_meta yet")